# 04 - Statystyki i Wizualizacja

Ten notebook obejmuje:
1. ewaluacje mIoU i IoU per klasa,
2. inference na real screenshotach z overlay,
3. dashboard metryk z MLflow.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image
import cv2
import mlflow
import torch

PROJECT_ROOT = Path.cwd().resolve()
import sys
if not (PROJECT_ROOT / 'src').exists():
    p = PROJECT_ROOT
    while True:
        if (p / 'src').exists():
            PROJECT_ROOT = p
            break
        if p == p.parent:
            break
        p = p.parent
sys.path.insert(0, str(PROJECT_ROOT))

TEST_DIR = PROJECT_ROOT / 'data' / 'segmentation' / 'test'
REAL_SCREENSHOTS_DIR = PROJECT_ROOT / 'src' / 'dataset' / 'Real_screenshots'
OUT_DIR = PROJECT_ROOT / 'data' / 'real_screenshot_predictions'
MODEL_PATH = PROJECT_ROOT / 'models' / 'segmentation_unet_combined.pt'
MLFLOW_DB = (PROJECT_ROOT / 'mlruns' / 'mlflow.db').resolve()
MLFLOW_TRACKING_URI = f'sqlite:///{MLFLOW_DB}'
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

from src.models.segmentation_detector import (
    SegmentationDetector, SegmentationDataset,
    extract_instances, remap_instances_to_image,
    detect_playfield_bbox, parse_hud_numbers, ID_TO_CLASS,
)
from src.utils.mlflow_logger import MLflowLogger

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('MODEL_PATH:', MODEL_PATH)
print('MLflow tracking URI:', MLFLOW_TRACKING_URI)

In [ ]:
def compute_miou(detector, dataset, id_to_class):
    class_ids = sorted(id_to_class.keys())
    intersection = {cid: 0 for cid in class_ids}
    union = {cid: 0 for cid in class_ids}
    for idx in range(len(dataset)):
        img_t, mask_t = dataset[idx]
        img_np = (img_t.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
        pred = detector.predict_mask(img_np)
        gt = mask_t.numpy().astype(np.int32)
        for cid in class_ids:
            if cid == 0:
                continue
            pred_c = (pred == cid).astype(np.uint8)
            gt_c = (gt == cid).astype(np.uint8)
            inter = int((pred_c & gt_c).sum())
            uni = int((pred_c | gt_c).sum())
            intersection[cid] += inter
            union[cid] += uni

    ious = {}
    for cid in class_ids:
        if cid == 0:
            continue
        if union[cid] > 0:
            ious[id_to_class[cid]] = float(intersection[cid]) / float(union[cid])
        else:
            ious[id_to_class[cid]] = None
    vals = [v for v in ious.values() if v is not None]
    miou = float(np.mean(vals)) if vals else None
    return ious, miou

if not MODEL_PATH.exists():
    print('Model not found:', MODEL_PATH)
else:
    detector = SegmentationDetector.load(MODEL_PATH, device=DEVICE)
    test_ds = SegmentationDataset(TEST_DIR)
    per_class_iou, miou = compute_miou(detector, test_ds, ID_TO_CLASS)
    print('mIoU:', miou)

    with MLflowLogger(experiment_name='segmentation_training', run_name='segmentation_eval') as logger:
        logger.log_params({'model_path': str(MODEL_PATH), 'dataset': str(TEST_DIR)})
        metric_payload = {'miou': float(miou) if miou is not None else np.nan}
        for label, value in per_class_iou.items():
            metric_payload[f'iou_{label}'] = float(value) if value is not None else np.nan
        logger.log_metrics(metric_payload)

    plot_items = [(k, v) for k, v in per_class_iou.items() if v is not None]
    if plot_items:
        labels, values = zip(*plot_items)
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.bar(labels, values, color='steelblue')
        ax.set_ylim(0, 1)
        ax.set_ylabel('IoU')
        ax.set_title(f'Per-class IoU (mIoU={miou:.3f})')
        ax.tick_params(axis='x', rotation=45)
        plt.tight_layout()
        plt.show()

In [ ]:
def _group_for_label(label: str) -> str:
    if label in {'blinky', 'pinky', 'inky', 'clyde', 'frightened_ghost', 'ghost_eyes'}:
        return 'ghosts'
    if label == 'fruit':
        return 'fruit'
    if label == 'pacman':
        return 'pacman'
    if label in {'pellet', 'power_pellet'}:
        return 'collectibles'
    return 'other'

GROUP_COLORS = {
    'pacman': '#ffd400',
    'ghosts': '#5ac8fa',
    'fruit': '#ff7a59',
    'collectibles': '#b8f397',
    'other': '#ffffff',
}

def show_annotated_image(image: np.ndarray, instances: list, playfield_bbox=None, figsize=(14, 12), show=True):
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    ax.imshow(image)
    if playfield_bbox is not None:
        x0, y0, bw, bh = playfield_bbox
        ax.add_patch(Rectangle((x0, y0), bw, bh, fill=False, edgecolor='#00e5ff', linewidth=3))
    for obj in instances:
        x, y, w, h = obj['bbox']
        label = obj['label']
        color = GROUP_COLORS.get(_group_for_label(label), '#ffffff')
        ax.add_patch(Rectangle((x, y), w, h, fill=False, edgecolor=color, linewidth=2))
        ax.text(x, max(0, y - 6), label, color=color, fontsize=9, weight='bold',
                bbox=dict(facecolor='black', alpha=0.6, pad=1, edgecolor='none'))
    ax.axis('off')
    if show:
        plt.show()
    return fig, ax

if not MODEL_PATH.exists():
    print('Model not found:', MODEL_PATH)
else:
    detector = SegmentationDetector.load(MODEL_PATH, device=DEVICE)
    img_path = REAL_SCREENSHOTS_DIR / 'Screenshot_20260614_131655.png'
    image = np.asarray(Image.open(img_path).convert('RGB'), dtype=np.uint8)
    playfield_bbox = detect_playfield_bbox(image)
    x0, y0, bw, bh = playfield_bbox
    crop = image[y0:y0+bh, x0:x0+bw]
    resized = cv2.resize(crop, (224, 248), interpolation=cv2.INTER_AREA)
    pred_mask = detector.predict_mask(resized)

    instances_local = extract_instances(pred_mask, ID_TO_CLASS, min_area=10)
    instances = remap_instances_to_image(instances_local, playfield_bbox, pred_mask.shape)

    outdir = OUT_DIR / img_path.stem
    outdir.mkdir(parents=True, exist_ok=True)
    fig, _ = show_annotated_image(image, instances, playfield_bbox=playfield_bbox, show=True)
    fig.savefig(outdir / 'annotated_overlay.png', dpi=150, bbox_inches='tight')
    plt.close(fig)
    print('Saved overlay:', outdir / 'annotated_overlay.png')

In [ ]:
client = mlflow.tracking.MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)
exp = client.get_experiment_by_name('segmentation_training')

if exp is None:
    print("Experiment 'segmentation_training' not found")
else:
    runs = client.search_runs(
        experiment_ids=[exp.experiment_id],
        order_by=['attributes.start_time DESC'],
        max_results=30,
    )

    rows = []
    for r in runs:
        rows.append({
            'run_name': r.data.tags.get('mlflow.runName', r.info.run_id[:8]),
            'status': r.info.status,
            'start_time': pd.to_datetime(r.info.start_time, unit='ms'),
            'train_loss': r.data.metrics.get('train_loss', np.nan),
            'val_loss': r.data.metrics.get('val_loss', np.nan),
            'best_val_loss': r.data.metrics.get('best_val_loss', np.nan),
            'miou': r.data.metrics.get('miou', np.nan),
            'n_instances': r.data.metrics.get('n_instances', np.nan),
        })

    df_runs = pd.DataFrame(rows)
    display(df_runs.head(20))

    if not df_runs.empty:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        train_df = df_runs.dropna(subset=['best_val_loss']).copy().sort_values('start_time')
        if not train_df.empty:
            axes[0].plot(train_df['start_time'], train_df['best_val_loss'], marker='o', color='tab:blue')
            axes[0].set_title('Best val loss over time')
            axes[0].set_ylabel('best_val_loss')
            axes[0].tick_params(axis='x', rotation=30)

        eval_df = df_runs.dropna(subset=['miou']).copy().sort_values('start_time')
        if not eval_df.empty:
            axes[1].plot(eval_df['start_time'], eval_df['miou'], marker='o', color='tab:green')
            axes[1].set_title('mIoU over time')
            axes[1].set_ylabel('mIoU')
            axes[1].set_ylim(0, 1)
            axes[1].tick_params(axis='x', rotation=30)

        plt.tight_layout()
        plt.show()